[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance4_cours.ipynb)

# Séance 3.4 — Régression linéaire — expliquer, et de combien

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'étude de cas en binôme)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- ajuster une régression avec `smf.ols("y ~ x", donnees).fit()`
- lire un coefficient, sa p-value et son intervalle de confiance
- dire ce que le R² mesure — et ce qu'il ne mesure pas
- interpréter un coefficient « toutes choses égales par ailleurs »
- faire entrer une ou plusieurs variables qualitatives dans un modèle
- reconnaître une extrapolation et refuser d'y répondre

## De « il y a un lien » à « de combien »

En séance 3.3, vous avez établi que le nombre de produits distincts (`nart`) et le montant
d'une commande sont liés. Le directeur commercial pose la question suivante :

> *« Si nos vendeurs poussent **un produit distinct de plus** dans chaque panier,
> ça rapporte combien ? »*

Une corrélation ne peut pas répondre : c'est un nombre sans unité. La
**régression** répond en euros.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

print(cmd.shape)
cmd.head(3)

## 1. Une droite qui résume le nuage

La régression cherche la droite qui passe **au plus près** de tous les points.
Elle s'écrit avec un `~`, qui se lit « expliquée par » : `"ca ~ nart"`.

Deux étapes, et la distinction compte : `smf.ols(...)` **construit** le
modèle, `.fit()` l'**ajuste** aux données. Oublier le second est l'erreur que
tout le monde commet une fois — la cellule suivante est volontairement fausse.

In [ ]:
smf.ols("ca ~ nart", cmd).summary()   ## erreur volontaire : il manque .fit()

Dernière ligne :

```
AttributeError: 'OLS' object has no attribute 'summary'
```

Traduction : l'objet rendu par `smf.ols(...)` est un modèle **non ajusté**. Il
n'a aucun résultat à résumer, puisqu'il n'a pas encore regardé les données.
**Seule la dernière ligne compte** — et la réparation tient en cinq
caractères.

Avec `.fit()`, cette fois :

In [ ]:
# Le ~ se lit "expliquee par" ; .fit() ajuste effectivement la droite
m = smf.ols("ca ~ nart", cmd).fit()

print(m.summary().tables[1])   ## le tableau des coefficients

Deux lignes, six colonnes. Pour l'instant, regardez-en **deux** :

- **`coef` de `nart` : 15,93.** Chaque produit distinct supplémentaire dans un panier
  correspond à **+15,93 € de commande**. Voilà la réponse au directeur
  commercial.
- **`P>|t|` : 0,000.** La p-value de la séance 3.2, appliquée au coefficient.
  Sous 0,05 : ce coefficient n'est pas un artefact du hasard.

L'`Intercept` (221,95 €) est le montant prédit pour une commande de **zéro**
produit. Aucun sens commercial — c'est normal, l'intercept sert à caler la
droite, pas à être interprété.

### La droite, sur le nuage

Ces deux nombres suffisent à tracer la droite : l'`Intercept` donne sa hauteur
en 0, le coefficient donne sa pente. Regardons-la.

In [ ]:
# La droite du modele, calculee sur une grille de valeurs de nart
grille = pd.DataFrame({"nart": range(0, 260, 10)})   ## jusqu'au maximum observe
grille["pred"] = m.predict(grille)   ## 221,95 + 15,93 * nart

cmd.plot(kind="scatter", x="nart", y="ca", alpha=0.3, s=8, figsize=(7, 4))
plt.plot(grille["nart"], grille["pred"], color="red", linewidth=2)
plt.ylim(0, 5000)   ## sans quoi une commande a 16 775 EUR ecrase tout
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.title("ca ~ nart : la droite ajustee")
plt.show()

**Voilà le modèle.** La droite rouge, c'est `ca = 221,95 + 15,93 × nart` —
rien de plus. Elle part de 222 € à gauche, et monte de 15,93 € chaque fois
qu'on avance d'un produit distinct.

« Au plus près » a un sens précis : parmi toutes les droites possibles, la
régression retient celle qui rend la **somme des écarts verticaux au carré**
la plus petite. Ces écarts, les voici — sur trente commandes, pour y voir clair.

In [ ]:
ech = cmd.sample(30, random_state=0).copy()   ## 30 commandes au hasard
ech["pred"] = m.predict(ech)                 ## ce que la droite annonce

plt.figure(figsize=(7, 4))
plt.vlines(ech["nart"], ech["pred"], ech["ca"], color="grey")   ## les ecarts
plt.scatter(ech["nart"], ech["ca"], color="#2878B5", s=30, zorder=5)
plt.plot(grille["nart"], grille["pred"], color="red", linewidth=2)
plt.xlim(0, 120); plt.ylim(0, 3000)
plt.xlabel("nart : nombre de produits distincts"); plt.ylabel("ca (euros)")
plt.title("La droite rend la somme de ces ecarts, au carre, la plus petite")
plt.show()

**Trente commandes, trente traits gris.** Chaque trait va du point observé
jusqu'à ce que la droite annonçait : c'est la part que le modèle n'explique pas.
Le plus long fait ici **1 099 €**.

Au carré, pour deux raisons : pour que les traits du haut n'annulent pas ceux du
bas, et pour qu'un trait deux fois plus long compte **quatre** fois plus. Ce sont
donc les traits les plus longs qui décident de la pente — une seule commande très
atypique peut faire pivoter la droite entière.

La droite décrit une **tendance moyenne**, pas une commande particulière — et
la section suivante met un chiffre sur l'épaisseur du nuage qui l'entoure.

### Les quatre autres colonnes

| Colonne | Ce qu'elle dit |
|---|---|
| `std err` | l'incertitude de mesure : avec une autre année de commandes, la pente aurait bougé d'environ **0,87 €** |
| `t` | le coefficient rapporté à cette incertitude — ici 18 fois plus grand qu'elle ; c'est de là que sort la p-value |
| `[0.025` `0.975]` | l'intervalle de confiance : le vrai effet est plausiblement entre **14,22 €** et **17,65 €** |

> 📖 **L'erreur type (`std err`).** C'est l'incertitude de mesure du
> coefficient. Cette pente n'a été estimée que sur 1 955 commandes ; sur une
> autre année on aurait trouvé un peu autre chose, et `std err` dit de combien.
> C'est la même idée que la fourchette de la séance 3.2, obtenue cette fois par
> une formule plutôt que par rééchantillonnage — et l'intervalle de confiance
> de la ligne suivante en découle directement.
> [Wikipédia](https://fr.wikipedia.org/wiki/Erreur_type)

Un coefficient s'annonce toujours avec sa fourchette : « environ 16 €, entre
14 et 18 ». Jamais « 15,9344 € ».

## 2. Le R² — ce qu'il dit, ce qu'il ne dit pas

In [ ]:
print("R2 :", round(m.rsquared, 3))   ## la part de variation reproduite

In [ ]:
mq = smf.ols("ca ~ qte", cmd).fit()   ## meme cible, une autre variable
g2 = pd.DataFrame({"qte": range(0, 6500, 100)})

fig, ax = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
ax[0].scatter(cmd["nart"], cmd["ca"], s=6, alpha=0.25, color="grey")
ax[0].plot(grille["nart"], grille["pred"], color="red", linewidth=2)
ax[1].scatter(cmd["qte"], cmd["ca"], s=6, alpha=0.25, color="grey")
ax[1].plot(g2["qte"], mq.predict(g2), color="red", linewidth=2)
ax[0].set_title("ca ~ nart : R2 = 0,146", fontsize=10)
ax[1].set_title("ca ~ qte : R2 = 0,719", fontsize=10)
ax[0].set_ylim(0, 5000); ax[0].set_xlim(0, 260); ax[1].set_xlim(0, 6500)
ax[0].set_ylabel("ca (euros)"); ax[0].set_xlabel("nart"); ax[1].set_xlabel("qte")
plt.show()

**Le R², c'est cette épaisseur-là.** Les deux nuages ont la même échelle
verticale et la même cible. À gauche, à 50 produits distincts, on trouve des
commandes de 200 € comme de 3 000 € : la droite passe au milieu d'un nuage qui
l'ignore. À droite, les points se serrent autour de la droite.

**0,146 contre 0,719.** Ce n'est pas que la droite de gauche soit fausse — son
coefficient est mesuré au demi-euro près. C'est qu'elle ne suffit pas à prédire.
Coefficient et R² répondent à deux questions différentes : « de combien ? » et
« est-ce que ça suffit ? ».

> ⚠️ Les échelles horizontales diffèrent — produits distincts à gauche, unités à
> droite. Ce qui se compare ici, c'est la **dispersion verticale**, pas la largeur.

**0,146.** Le nombre de produits distincts reproduit **15 %** de la variation des
montants. Les 85 % restants viennent d'ailleurs : le prix unitaire des
produits, les
quantités, le type de client.

> ⚠️ **Un R² faible ne rend pas le coefficient faux.** Ici : 15 % d'explication
> et un effet mesuré à ±1,7 € près. Ce sont deux questions différentes :
>
> - le **coefficient** répond à « de combien ? »
> - le **R²** répond à « est-ce que ma variable suffit à prédire ? »
>
> On peut très bien mesurer précisément un effet réel mais petit.

## 3. Plusieurs variables — et le coefficient qui change

Ajoutons le nombre d'unités (`qte`). Avant d'exécuter : à votre avis, le coefficient
de `nart` va-t-il monter, descendre, ou rester à 15,93 ?

In [ ]:
m2 = smf.ols("ca ~ nart + qte", cmd).fit()   ## deux variables, un +

# Version etroite du tableau : deux colonnes suffisent pour l'essentiel
pd.DataFrame({"coef": m2.params.round(2), "p": m2.pvalues.round(3)})

**15,93 € → 2,05 €.** Divisé par huit, sans qu'on ait touché aux données.

Pourquoi ? Les commandes qui portent sur beaucoup de produits distincts contiennent
aussi beaucoup d'unités. Dans le premier modèle, `nart` récupérait **tout
le mérite** : le sien et celui des quantités. Le second modèle sépare les
deux.

Les deux coefficients sont justes. Ils ne répondent simplement pas à la même
question :

| Modèle | Ce que dit le coefficient de `nart` |
|---|---|
| `ca ~ nart` | une commande avec un produit distinct de plus vaut 15,93 € de plus |
| `ca ~ nart + qte` | **à nombre d'unités égal**, un produit distinct de plus vaut 2,05 € |

> ⚠️ **Un coefficient ne se lit jamais seul.** Il se lit « toutes choses égales
> par ailleurs », et le « par ailleurs » est exactement la liste des variables
> du modèle. Deux études sérieuses peuvent publier des chiffres différents pour
> cette raison, sans qu'aucune ne se trompe.

Et le R² ?

In [ ]:
print("R2 simple  :", round(m.rsquared, 3))    ## 0,146
print("R2 complet :", round(m2.rsquared, 3))   ## 0,721 : qte apportait beaucoup

De 0,146 à 0,721. La deuxième variable apportait vraiment quelque chose.

## 4. Des variables qualitatives dans le modèle

`pays` est du texte. La régression le transforme en comparaisons : une
modalité sert de **référence**, les autres se lisent par rapport à elle.

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

# pays est du texte : une modalite devient la reference, les autres
# se lisent en ecart par rapport a elle
m3 = smf.ols("ca ~ qte + pays", sub).fit()
pd.DataFrame({"coef": m3.params.round(2), "p": m3.pvalues.round(3)})

L'Allemagne n'apparaît pas : c'est la **référence** (première par ordre
alphabétique). Tout se lit par rapport à elle, à nombre d'unités égal :

- **Royaume-Uni : −99,59 €, p = 0,001.** À nombre d'unités identique, une
  commande britannique vaut cent euros de moins qu'une allemande. Solide.
- **France : −27,32 €, p = 0,448.** Rien ne distingue la France de
  l'Allemagne — c'est très exactement la conclusion de la séance 3.2, obtenue
  cette fois en tenant compte des quantités.
- **Irlande : +34,35 €, p = 0,346.** Le fameux écart irlandais **disparaît**
  une fois les quantités prises en compte : les commandes irlandaises ne sont
  pas plus chères par produit, elles sont simplement **plus grosses**.

C'est le genre de phrase qu'on ne peut écrire qu'avec une régression.

### Deux variables qualitatives à la fois

Rien n'empêche d'en mettre plusieurs. Ajoutons `jour`, le jour de la semaine.

In [ ]:
# Deux qualitatives : chacune recoit SA propre modalite de reference
m4 = smf.ols("ca ~ qte + pays + jour", sub).fit()

pd.DataFrame({"coef": m4.params.round(2), "p": m4.pvalues.round(3)})

Le tableau a maintenant **deux blocs** d'étiquettes, `pays[T.…]` et
`jour[T.…]`. Trois règles pour le lire.

**1. Chaque variable qualitative a sa propre référence.** L'Allemagne pour
`pays`, le dimanche pour `jour` — les premières par ordre alphabétique dans
chaque variable. Elles n'apparaissent pas dans le tableau : elles *sont* le
point de comparaison.

**2. L'`Intercept` correspond à la combinaison de toutes les références.**
Ici : une commande allemande, passée un dimanche, avec `qte` = 0. Comme
toujours, il cale le modèle et ne s'interprète pas.

**3. Chaque coefficient se lit toutes les autres variables tenues fixes.**
Donc `pays[T.Royaume-Uni]` = −94,66 € se lit désormais : *à nombre d'unités
égal **et pour un même jour de la semaine**, une commande britannique vaut 95 €
de moins qu'une allemande*. Le « par ailleurs » s'est allongé d'un cran.

### Une variable qualitative se juge en bloc

Regardez les cinq lignes de `jour` : p vaut 0,706, 0,997, 0,714, 0,423 et
0,380. Aucune ne s'approche de 0,05.

Ce n'est **pas** cinq résultats séparés, c'est un seul : rien ne distingue les
jours entre eux. Une variable qualitative se lit comme un **bloc** — on ne
retient pas « le vendredi est à +35,87 € » au motif qu'il a le plus gros
coefficient. C'est d'ailleurs le piège des tests répétés de la séance 3.2 :
avec cinq modalités, on regarde cinq p-values d'un coup.

Le R² le confirme d'un autre côté :

In [ ]:
print("R2 sans jour :", round(m3.rsquared, 3))   ## ca ~ qte + pays
print("R2 avec jour :", round(m4.rsquared, 3))   ## ca ~ qte + pays + jour

**0,792 contre 0,793.** Cinq coefficients de plus pour un millième de R² : le
jour de la semaine n'apporte rien.

> ⚠️ **Chaque qualitative coûte cher en lisibilité.** Une variable à *k*
> modalités ajoute *k − 1* lignes au tableau. `pays` sur les 23 pays du
> fichier en ajouterait 22. Ajoutez-en une parce qu'une question métier la
> demande, jamais « pour voir ».

## 5. Prédire — et jusqu'où

In [ ]:
nouvelles = pd.DataFrame({"nart": [20]})   ## meme nom que dans la formule

print("commande a 20 produits distincts :", m.predict(nouvelles).round(2).iloc[0], "euros")

540,63 € — soit `221,95 + 20 × 15,93`. Une régression est une machine à
prédire autant qu'à expliquer.

Maintenant, la même machine, hors du domaine observé.

In [ ]:
absurde = pd.DataFrame({"nart": [500]})   ## bien au-dela de l'observe

print("commande a 500 produits distincts :", m.predict(absurde).round(2).iloc[0], "euros")
print("maximum reellement observe :", cmd["nart"].max(), "produits distincts")

**8 189 €**, annoncés sans la moindre réserve — pour une commande de 500
produits distincts alors que la plus fournie du fichier en compte **259**.

Le modèle a prolongé sa droite dans une zone où **il n'a jamais rien vu**.
Rien dans les données ne dit que la relation reste droite là-bas ; elle
pourrait plafonner, ou s'effondrer.

### Les deux prédictions, sur le nuage

Le plus simple est de les placer sur la figure de la section 1, en prolongeant
la droite jusqu'à 500.

In [ ]:
grille = pd.DataFrame({"nart": range(0, 501, 10)})   ## prolongee jusqu'a 500
grille["pred"] = m.predict(grille)

pts = pd.DataFrame({"nart": [20, 500]})   ## la prediction sage, et l'absurde
pts["pred"] = m.predict(pts)

maxi = cmd["nart"].max()   ## 259 : la ou s'arretent les donnees
print(pts.round(2))

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(cmd["nart"], cmd["ca"], alpha=0.2, s=8, label="commandes observees")
plt.plot(grille["nart"], grille["pred"], color="red", label="droite du modele")
plt.axvspan(maxi, 500, color="red", alpha=0.12)   ## zone jamais observee
plt.scatter(pts["nart"], pts["pred"], color="black", s=50, zorder=5,
            label="predictions")

plt.ylim(0, 9000)
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.legend(fontsize=8)
plt.show()

Les deux points noirs sont les deux prédictions, et tout est dans leur
voisinage.

- Celui de gauche, à `nart` = 20, tombe **au milieu du nuage**. Des centaines
  de vraies commandes l'entourent, au-dessus et en dessous. Le modèle
  interpole : il propose une valeur moyenne là où il a vu des données, et ce
  chiffre se défend.
- Celui de droite, à `nart` = 500, flotte dans la **bande rouge**, où le
  fichier ne contient pas un seul point. La droite continue parce qu'une
  droite continue toujours, pas parce que quelque chose la soutient.

La bande commence à 259, le maximum réellement observé. Tout ce qui est à sa
droite est une **hypothèse**, pas une mesure — et le modèle ne fait aucune
différence entre les deux.

> ⚠️ **L'extrapolation est l'erreur silencieuse par excellence.** Le résultat
> a l'air d'un résultat. Avant toute prédiction, comparez vos valeurs d'entrée
> au `min` et au `max` observés — c'est le seul moyen de savoir de quel côté
> de la bande vous vous trouvez.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| ajuster un modèle | `m = smf.ols("ca ~ nart", cmd).fit()` |
| le tableau des coefficients | `print(m.summary().tables[1])` |
| une version étroite | `pd.DataFrame({"coef": m.params.round(2), "p": m.pvalues.round(3)})` |
| le R² | `m.rsquared` |
| plusieurs variables | `smf.ols("ca ~ nart + qte", cmd)` |
| une variable qualitative | `smf.ols("ca ~ qte + pays", cmd)` |
| deux qualitatives | `smf.ols("ca ~ qte + pays + jour", cmd)` — une référence chacune |
| tracer la droite ajustée | `m.predict(grille)` puis `plt.plot(...)` |
| prédire | `m.predict(pd.DataFrame({"nart": [20]}))` |

## Lire un tableau de régression

| Colonne | Ce qu'elle dit |
|---|---|
| `coef` | de combien `y` bouge quand `x` augmente d'une unité, **les autres variables restant fixes** |
| `std err` | l'incertitude de mesure : avec une autre année de commandes, la pente aurait bougé d'environ **0,87 €** |
| `P>|t|` | la p-value : sous 0,05, on retient le coefficient |
| `[0.025 0.975]` | l'intervalle de confiance du coefficient |

## Les cinq phrases à retenir

1. **Un coefficient se lit toujours « à autres variables constantes ».** Seul,
   `nart` valait 15,93 € ; avec `qte` dans le modèle, 2,05 €. Les deux sont
   justes, ils ne répondent pas à la même question.

2. **Un R² faible n'invalide pas un coefficient.** `ca ~ nart` explique 15 %
   de la variation et son coefficient est parfaitement mesuré.

3. **Une régression ne démontre pas une causalité.** Elle mesure une
   association, en tenant compte des variables qu'on lui a données — et
   d'aucune autre.

4. **Une variable qualitative se juge en bloc.** Ses *k − 1* coefficients
   sont un seul résultat, pas *k − 1* résultats : si aucune modalité ne sort,
   c'est la variable entière qui n'apporte rien.

5. **Hors du domaine observé, un modèle invente.** Il répondra quand même,
   sans prévenir.